In [4]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.

In [5]:
import os
import gc
import ast
import shutil
import random
import sys

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import FileLink

import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchvision
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import RetinaNetHead
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.models.detection.anchor_utils import AnchorGenerator

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix
import torchmetrics
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [6]:
def find_dataset_paths():
    input_root = '/kaggle/input'
    csv_path, train_dir = None, None
    for root, dirs, files in os.walk(input_root):
        for file in files:
            if file.endswith('.csv') and 'submission' not in file:
                try:
                    df = pd.read_csv(os.path.join(root, file), nrows=1)
                    if 'patientId' in df.columns and ('Target' in df.columns or 'boxes' in df.columns):
                        csv_path = os.path.join(root, file)
                        break
                except: continue
        if csv_path: break
    
    max_files = 0
    for root, dirs, files in os.walk(input_root):
        img_count = len([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if img_count > 500 and img_count > max_files:
            max_files = img_count
            train_dir = root
    return csv_path, train_dir

class Config:
    try: CSV_FILE, TRAIN_DIR = find_dataset_paths()
    except: CSV_FILE, TRAIN_DIR = "", ""

    FINAL_WEIGHT_PATH = "/kaggle/input/weight1/retinanet_best_map50.pth"

    SEED = 42
    DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    NUM_WORKERS = 4

    IMG_SIZE = 512
    BATCH_SIZE = 8 
   
    NUM_EPOCHS = 30         
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4

    USE_AMP = True
    NUM_CLASSES = 2

    # Đường dẫn lưu model
    CHECKPOINT_DIR = './checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(Config.SEED)

In [7]:
def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    return images, list(targets)

In [8]:
def get_transforms():
    return A.Compose([
        A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        # Giới hạn xoay 10 độ như yêu cầu
        A.ShiftScaleRotate(rotate_limit=10, scale_limit=0.1, shift_limit=0.1, p=0.5), 
        # Tăng giảm sáng tối 10%
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

def get_val_transforms():
    return A.Compose([
        A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

In [9]:
class RSNADataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform
        
        # Check đuôi file ảnh
        self.ext = '.png'
        if root_dir and os.path.exists(root_dir) and len(os.listdir(root_dir)) > 0:
            sample = [f for f in os.listdir(root_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))][0]
            self.ext = os.path.splitext(sample)[1]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, f"{row['patientId']}{self.ext}")
        
        # 1. Load ảnh gốc
        image = cv2.imread(img_path)
        if image is None: image = np.zeros((Config.IMG_SIZE, Config.IMG_SIZE, 3), dtype=np.uint8)
        else: image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 2. Load Box (Xử lý chuỗi nếu cần)
        raw_boxes = row['boxes']
        if isinstance(raw_boxes, str): raw_boxes = ast.literal_eval(raw_boxes)
        
        boxes_conv = []
        labels_conv = []
        if isinstance(raw_boxes, list):
            for box in raw_boxes:
                if len(box) == 4:
                    x, y, w, h = box
                    if w > 0 and h > 0:
                        boxes_conv.append([x, y, x+w, y+h]) # Pascal VOC: x_min, y_min, x_max, y_max
                        labels_conv.append(1) # Class 1: Pneumonia
        
        if len(boxes_conv) == 0: 
            boxes_conv = []
            labels_conv = []

        # 3. Augmentation
        if self.transform:
            try:
                t = self.transform(image=image, bboxes=boxes_conv, labels=labels_conv)
                image = t['image']
                boxes_conv = t['bboxes']
                labels_conv = t['labels']
            except: # Fallback nếu augmentation lỗi
                trans = ToTensorV2()
                t = trans(image=image)
                image = t['image']
                boxes_conv, labels_conv = [], []

        # 4. Target Dict
        target = {
            'boxes': torch.tensor(boxes_conv, dtype=torch.float32) if len(boxes_conv) > 0 else torch.zeros((0, 4), dtype=torch.float32),
            'labels': torch.tensor(labels_conv, dtype=torch.int64) if len(labels_conv) > 0 else torch.zeros((0,), dtype=torch.int64),
            'image_id': torch.tensor([idx])
        }
        return image, target

In [10]:
def get_retinanet_model():

    anchor_sizes = ((32,), (64,), (128,), (256,), (384,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)

    model = retinanet_resnet50_fpn(
        weights=None, 
        weights_backbone=None, 
        trainable_backbone_layers=5,
        anchor_generator=anchor_generator
    )

    try: num_anchors = model.head.classification_head.num_anchors
    except AttributeError: num_anchors = 9
    model.head = RetinaNetHead(256, num_anchors, Config.NUM_CLASSES)
    
    model.transform = GeneralizedRCNNTransform(
        min_size=Config.IMG_SIZE, max_size=Config.IMG_SIZE,
        image_mean=[0.0, 0.0, 0.0], image_std=[1.0, 1.0, 1.0],
        fixed_size=(Config.IMG_SIZE, Config.IMG_SIZE)
    )
    return model

In [11]:
def train_one_epoch(model, loader, optimizer, scaler, device):
    model.train()
    total_loss = 0
    
    for inputs, targets in tqdm(loader, desc="Training"):
        # Single Stream Input
        inputs = list(img.to(device) for img in inputs)
        
        # Target xử lý list of dicts
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        with autocast(enabled=Config.USE_AMP):
            loss_dict = model(inputs, targets)
            losses = sum(loss for loss in loss_dict.values())
            
        optimizer.zero_grad()
        scaler.scale(losses).backward()
        
        # Unscale & Clip Grad (Giữ lại từ kinh nghiệm trước để tránh lỗi mAP=0)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses.item()
        
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_gpu(model, loader, device):
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=False)
    metric.to(device)
    for imgs, tgts in tqdm(loader, desc="Evaluating", leave=False):
        imgs = list(img.to(device) for img in imgs)
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
        preds = model(imgs)
        metric.update(preds, tgts)
    result = metric.compute()
    return result['map'].item(), result['map_50'].item()

In [12]:
def main():
    torch.cuda.empty_cache()
    gc.collect()
    print(f" Training Config: {Config.DEVICE} | Epochs: {Config.NUM_EPOCHS} | Batch: {Config.BATCH_SIZE}")
    
    if not Config.CSV_FILE:
        print(" Lỗi: Không tìm thấy Data"); return

    # --- Load Data & Split (Giống hệt các model khác) ---
    df_raw = pd.read_csv(Config.CSV_FILE)
    cols = df_raw.columns
    col_id = 'patientId' if 'patientId' in cols else cols[0]
    
    patient_data = []
    if 'boxes' in cols:
        for _, row in df_raw.iterrows():
            patient_data.append({'patientId': row[col_id], 'Target': row.get('Target', 0), 'boxes': row['boxes']})
    else:
        for pid, grp in df_raw.groupby(col_id):
            target = 1 if (grp.get('Target', 0) == 1).any() else 0
            boxes = []
            if target == 1:
                for _, r in grp.iterrows():
                    if 'x' in r and not pd.isna(r['x']): boxes.append([r['x'], r['y'], r['width'], r['height']])
            patient_data.append({'patientId': pid, 'Target': target, 'boxes': boxes})
            
    meta_df = pd.DataFrame(patient_data)

    tr_meta, val_meta = train_test_split(meta_df, test_size=0.2, stratify=meta_df['Target'], random_state=Config.SEED)
    tr_final = tr_meta.reset_index(drop=True)
    val_final = val_meta.reset_index(drop=True)
    
    print(f"Train Size: {len(tr_final)} | Val Size: {len(val_final)}")

    tr_loader = DataLoader(RSNADataset(tr_final, Config.TRAIN_DIR, transform=get_transforms()), 
                           batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS, collate_fn=collate_fn)
    val_loader = DataLoader(RSNADataset(val_final, Config.TRAIN_DIR, transform=get_val_transforms()), 
                           batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, collate_fn=collate_fn)

    model = get_retinanet_model()
    model.to(Config.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.NUM_EPOCHS, eta_min=1e-6)
    
    scaler = GradScaler(enabled=Config.USE_AMP)
    
    best_map = 0.0
    history = {'epoch': [], 'loss': [], 'mAP': [], 'mAP_50': []}
    
    print("\n Bắt đầu huấn luyện...")
    for epoch in range(1, Config.NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, tr_loader, optimizer, scaler, Config.DEVICE)
        scheduler.step()
        
        # Evaluate
        map_coco, map_50 = evaluate_gpu(model, val_loader, Config.DEVICE)
        
        print(f"Ep {epoch} | Loss: {train_loss:.4f} | mAP: {map_coco:.4f} | mAP@50: {map_50:.4f}")
        
        history['epoch'].append(epoch)
        history['loss'].append(train_loss)
        history['mAP'].append(map_coco)
        history['mAP_50'].append(map_50)
        pd.DataFrame(history).to_csv(os.path.join(Config.CHECKPOINT_DIR, 'log.csv'), index=False)
        
        if map_coco > best_map:
            best_map = map_coco
            torch.save(model.state_dict(), os.path.join(Config.CHECKPOINT_DIR, "retinanet_best_map50.pth"))
            print("Saved Best Model")

In [13]:
#if __name__ == '__main__':
#    main()

In [14]:
FOLDER_TO_DOWNLOAD = '/kaggle/working/' 
OUTPUT_FILENAME = 'retinanet_weights' 

def zip_and_download_output():

    if not os.path.exists(FOLDER_TO_DOWNLOAD):
        print(f"Không tìm thấy thư mục '{FOLDER_TO_DOWNLOAD}'")
        return

    shutil.make_archive(OUTPUT_FILENAME, 'zip', FOLDER_TO_DOWNLOAD)
    
    zip_file = f"{OUTPUT_FILENAME}.zip"
    
    if os.path.exists(zip_file):
        print(f"Nén xong! Kích thước: {os.path.getsize(zip_file) / (1024*1024):.2f} MB")
        print("Link tải:")

        display(FileLink(zip_file))
    else:
        print("Không tạo được file zip.")



In [15]:
#zip_and_download_output()

In [16]:
def calculate_iou_xywh(boxA, boxB):

    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])

    interW = max(0.0, xB - xA)
    interH = max(0.0, yB - yA)
    interArea = interW * interH

    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]
    unionArea = boxAArea + boxBArea - interArea

    if unionArea <= 0:
        return 0.0
    return float(interArea / unionArea)

def get_score_at_threshold(gt_boxes, pred_boxes, threshold):
    if len(gt_boxes) == 0:
        return 1.0 if len(pred_boxes) == 0 else 0.0

    if len(pred_boxes) == 0:
        return 0.0

    pred_boxes = sorted(pred_boxes, key=lambda x: x[4], reverse=True)

    tp = 0
    fp = 0
    gt_matched = [False] * len(gt_boxes)

    for p_box in pred_boxes:
        p_bbox = p_box[:4]
        best_iou = 0.0
        best_gt_idx = -1

        for i, g_bbox in enumerate(gt_boxes):
            if gt_matched[i]:
                continue
            iou = calculate_iou_xywh(p_bbox, g_bbox)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i

        if best_iou >= threshold and best_gt_idx >= 0:
            tp += 1
            gt_matched[best_gt_idx] = True
        else:
            fp += 1

    fn = len(gt_boxes) - sum(gt_matched)
    return float(tp) / (tp + fp + fn)

def calculate_detection_metrics(gt_dict, pred_dict):
    rsna_thresholds = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75]
    rsna_scores = []
    map50_scores = []

    for img_id in gt_dict.keys():
        gts = gt_dict.get(img_id, [])
        preds = pred_dict.get(img_id, [])

        img_rsna_score = np.mean([get_score_at_threshold(gts, preds, t) for t in rsna_thresholds])
        rsna_scores.append(img_rsna_score)

        map50_scores.append(get_score_at_threshold(gts, preds, 0.5))

    return np.mean(rsna_scores), np.mean(map50_scores)

In [17]:
def calculate_classification_metrics(y_true, y_scores):

    if len(np.unique(y_true)) < 2:
        return 0.0, 0.0, 0.0, 0.0

    auc = roc_auc_score(y_true, y_scores)

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)

    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-6)
    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    y_pred_binary = (np.array(y_scores) >= best_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
    
    sensitivity = tp / (tp + fn + 1e-6)
    specificity = tn / (tn + fp + 1e-6)

    return auc, best_f1, sensitivity, specificity, best_threshold

In [18]:
def evaluate_all_metrics(ground_truth_dict, prediction_dict):
    y_true_cls = []
    y_pred_cls = []
    
    for img_id in ground_truth_dict.keys():
        gts = ground_truth_dict[img_id]
        y_true_cls.append(1 if len(gts) > 0 else 0)

        preds = prediction_dict.get(img_id, [])
        if len(preds) > 0:
            max_score = max([p[4] for p in preds])
            y_pred_cls.append(max_score)
        else:
            y_pred_cls.append(0.0)

    auc, f1, sens, spec, thres = calculate_classification_metrics(y_true_cls, y_pred_cls)

    rsna_map, map50 = calculate_detection_metrics(ground_truth_dict, prediction_dict)

    # 4. In kết quả
    print("\n" + "="*40)
    print(" BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ")
    print("="*40)
    print(f" [Object Detection] RSNA mAP (0.4-0.75): {rsna_map:.4f}")
    print(f" [Object Detection] mAP@0.5:            {map50:.4f}")
    print("-" * 40)
    print(f" [Classification] ROC AUC:              {auc:.4f}")
    print(f" [Classification] Best F1-Score:        {f1:.4f} (tại threshold={thres:.3f})")
    print(f" [Classification] Sensitivity (Recall): {sens:.4f}")
    print(f" [Classification] Specificity:          {spec:.4f}")
    print("="*40)
    
    return {
        "rsna_map": rsna_map,
        "map50": map50,
        "auc": auc,
        "f1": f1,
        "sensitivity": sens,
        "specificity": spec
    }

In [19]:
def xyxy_to_xywh(box):
    return [float(box[0]), float(box[1]), float(box[2]-box[0]), float(box[3]-box[1])]

def run_full_evaluation():

    if not Config.CSV_FILE or not os.path.exists(Config.CSV_FILE):
        print("Không tìm thấy file CSV dataset. Vui lòng kiểm tra lại đường dẫn.")
        return

    print("Đang xử lý dữ liệu gốc và tạo tập Validation...")
    df_raw = pd.read_csv(Config.CSV_FILE)

    patient_data = []
    for pid, grp in df_raw.groupby('patientId'):
        target = 1 if (grp.get('Target', 0) == 1).any() else 0
        boxes = []
        if target == 1:
            for _, r in grp.iterrows():
                if 'x' in r and not pd.isna(r['x']): 
                    boxes.append([r['x'], r['y'], r['width'], r['height']])
        patient_data.append({'patientId': pid, 'Target': target, 'boxes': boxes})
            
    meta_df = pd.DataFrame(patient_data)

    _, val_meta = train_test_split(
        meta_df, 
        test_size=0.2, 
        stratify=meta_df['Target'], 
        random_state=Config.SEED
    )
    
    print(f"   -> Tổng số bệnh nhân: {len(meta_df)}")
    print(f"   -> Validation Size:   {len(val_meta)}")

    val_dataset = RSNADataset(
        val_meta.reset_index(drop=True), 
        Config.TRAIN_DIR, 
        transform=get_val_transforms()
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=Config.BATCH_SIZE, 
        shuffle=False, 
        num_workers=Config.NUM_WORKERS, 
        collate_fn=collate_fn
    )

    print("Đang khởi tạo model và load weights...")
    model = get_retinanet_model()
    
    # Kiểm tra file weights
    if os.path.exists(Config.FINAL_WEIGHT_PATH):
        try:
            state_dict = torch.load(Config.FINAL_WEIGHT_PATH, map_location=Config.DEVICE)
            model.load_state_dict(state_dict)
            print(f"Đã load weights thành công từ: {Config.FINAL_WEIGHT_PATH}")
        except Exception as e:
            print(f"Lỗi khi load weights: {e}")
            return
    else:
        print(f"Lỗi: Không tìm thấy file weights tại {Config.FINAL_WEIGHT_PATH}")
        return

    model.to(Config.DEVICE)
    model.eval()

    ground_truth_dict = {}
    prediction_dict = {}
    
    print("Đang chạy đánh giá...")
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Evaluating"):
            images = list(img.to(Config.DEVICE) for img in images)
            outputs = model(images)
            
            for i, output in enumerate(outputs):
                img_id = str(targets[i]['image_id'].item())
                
                # --- 1. Lấy Ground Truth ---
                gt_boxes = targets[i]['boxes'].cpu().numpy()
                gts_xywh = [xyxy_to_xywh(box) for box in gt_boxes]
                ground_truth_dict[img_id] = gts_xywh
                pred_boxes_tensor = output['boxes']   # [N, 4]
                pred_scores_tensor = output['scores'] # [N]

                keep_indices = torchvision.ops.nms(pred_boxes_tensor, pred_scores_tensor, iou_threshold=0.3)

                final_boxes = pred_boxes_tensor[keep_indices]
                final_scores = pred_scores_tensor[keep_indices]

                pred_boxes_np = final_boxes.cpu().numpy() 
                pred_scores_np = final_scores.cpu().numpy()
                
                preds_formatted = []
                for box, score in zip(pred_boxes_np, pred_scores_np):
                    xywh = xyxy_to_xywh(box)
                    preds_formatted.append(xywh + [float(score)])
                
                prediction_dict[img_id] = preds_formatted

    print("Tính toán Metrics...")
    evaluate_all_metrics(ground_truth_dict, prediction_dict)

if __name__ == "__main__":
    run_full_evaluation()

Đang xử lý dữ liệu gốc và tạo tập Validation...
   -> Tổng số bệnh nhân: 26684
   -> Validation Size:   5337
Đang khởi tạo model và load weights...


/usr/local/lib/python3.11/dist-packages/torchvision/models/detection/backbone_utils.py:162: UserWarning: Changing trainable_backbone_layers has no effect if neither pretrained nor pretrained_backbone have been set to True, falling back to trainable_backbone_layers=5 so that all layers are trainable
  warnings.warn(


Đã load weights thành công từ: /kaggle/input/weight1/retinanet_best_map50.pth
Đang chạy đánh giá...


Evaluating:   0%|          | 0/668 [00:00<?, ?it/s]

Tính toán Metrics...

 BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ
 [Object Detection] RSNA mAP (0.4-0.75): 0.3537
 [Object Detection] mAP@0.5:            0.3748
----------------------------------------
 [Classification] ROC AUC:              0.8867
 [Classification] Best F1-Score:        0.6554 (tại threshold=0.173)
 [Classification] Sensitivity (Recall): 0.7121
 [Classification] Specificity:          0.8660
